[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter01_welcome_to_deep_learning/notebook.ipynb)

# Chapter 1: Welcome to Deep Learning

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

In the autumn of 2012, a small team from the University of Toronto entered a contest that most people outside computer vision had never heard of. The contest, called the ImageNet Large Scale Visual Recognition Challenge, handed every team about 1.2 million photographs, each one labelled with one of a thousand categories such as a tabby cat, a fire truck or a mushroom, and asked them to build a program that could name what was in photographs it had never seen. For years the best teams had been improving by a point or two at a time, using programs built on clever rules that experts had spent careers designing by hand, rules for spotting edges, corners and patches of texture.

The Toronto team, Alex Krizhevsky, Ilya Sutskever and their supervisor Geoffrey Hinton, did something that many experts at the time considered old fashioned. They gave the raw pixels of each photograph to a very large neural network with about 60 million adjustable numbers inside it, trained it for nearly a week on two ordinary gaming graphics cards, and let it work out for itself which patterns in the pixels were worth noticing. The contest scores each program by how often the right answer is missing from its top five guesses, and the second best team missed on 26.2 percent of the test photographs. The Toronto network, which came to be known as AlexNet, missed on only 15.3 percent.

A gap that large in a contest where progress usually came in small steps was a shock, and within two or three years almost every serious team in computer vision had switched to neural networks. The same ideas then spread to speech recognition, translation, medical imaging and, eventually, the chat assistants that millions of people now use every day. The announcement of those 2012 results is often described as the moment deep learning took over, and it is a good place for this book to begin.

What made AlexNet different was not that it followed better rules, because it was given no rules about images at all. It learned its own way of looking at a photograph, starting with simple patterns and building them up, layer by layer, into ideas as rich as a whisker or a wheel. That is the whole subject of this book, and by the end of it you will build, train and understand networks of your own. Before you read on, try to guess the answers to these four questions.

1. **Why could the experts' hand designed rules not keep up with a network that was given no rules at all?**
2. **What is a neural network actually made of, if you open it up?**
3. **Why might a network with many layers learn things that a network with one layer cannot?**
4. **If a network chooses its own features, what is left for you to do?**

## Where Machine Learning Runs Out of Road

Every model you trained in Books 4 and 5 had one thing in common, which is that it learned from columns that somebody had already prepared. A house came with its area, its number of bedrooms and its distance from the station, and the model's whole job was to work out how much each of those columns mattered. Book 5 then taught you the most valuable habit in classical machine learning, which is that a well built column, such as the rent per square metre, often does more for a model than any amount of tuning.

That habit has a hidden cost, and a tiny example shows exactly where it lies. Imagine a seed company that photographs a field from above and records where each plant is growing. The plants near the middle of the field are a healthy variety and the plants around the outside are a weaker one, and the company wants a model that can tell the two apart from nothing more than a plant's position. The first cell builds 400 such plants from a fixed seed, so you will see the same numbers that are printed here.

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=400, noise=0.08,
                    factor=0.45, random_state=7)
print(X[:3].round(2))
print(y[:3])

Each row of `X` holds two numbers, the plant's position across the field and its position up the field, and each value of `y` says which variety it is, with `1` for the healthy plants in the middle and `0` for the plants around the edge. Figure 1.1 shows all 400 plants, and you can see the pattern at a glance even though no single column reveals it.

> 🖼️ **Figure 1.1:** The 400 plants in the field. The healthy variety grows inside the dashed circle and the weaker variety grows around it, so no straight line can ever separate the two.

A logistic regression, the first classifier you met in Book 4, draws a single straight line through the field and calls everything on one side healthy and everything on the other side weak. The next cell splits the plants into a training set and a test set, exactly as you have always done, and measures how often the line is right on the plants it never saw.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7)
line = LogisticRegression().fit(X_train, y_train)
print(round(line.score(X_test, y_test), 2))

An accuracy of 0.5 on two equally common classes is exactly what you would get by tossing a coin, because there is no straight line that puts a circle on one side and a ring on the other. The model is not broken, since it is simply being asked a question that its shape cannot answer. Book 5 taught you the cure, which is to build a better column, and here the right column is how far each plant is from the middle of the field. The squared distance, the first number squared plus the second number squared, keeps the same order as the real distance and saves a square root.

In [ ]:
r_train = (X_train ** 2).sum(axis=1).reshape(-1, 1)
r_test = (X_test ** 2).sum(axis=1).reshape(-1, 1)
ring = LogisticRegression().fit(r_train, y_train)
print(round(ring.score(r_test, y_test), 2))

One new column took the same model from a coin toss to a perfect score, which is exactly the lesson of Book 5. Notice, though, where that column really came from. It came from you, a person who looked at Figure 1.1, saw a circle, and remembered that a circle is about distance from a centre. **Classical machine learning is only ever as good as the columns a person can think of, and that works beautifully until the data is something no person can turn into columns by hand.**

A field with two numbers per plant is easy to look at. A small colour photograph, only 224 pixels wide and 224 pixels tall, holds more than 150,000 numbers, and there is no picture you can draw of them that would show you the right column for spotting a cat. Table 1.1 shows how quickly the road runs out as the data moves further away from a tidy table.

**Table 1.1: As the data moves away from a tidy table, building good columns by hand becomes impossible.**

| Kind of data | What one example holds | Can a person build the columns? |
|---|---|---|
| Houses for sale | A dozen named measurements | Yes, easily |
| Plants in a field | Two positions | Yes, after looking at a plot |
| A small photograph | More than 150,000 pixel values | Not in any practical way |
| A paragraph of text | Hundreds of words in a set order | Only crudely, by counting words |

## What a Neural Network Is

A neural network is a model that builds its own columns. Instead of waiting for you to hand it the distance from the centre of the field, it starts from the raw positions and learns, through training, a set of new in between numbers that make the final decision easy. Those in between numbers are made by small units called **neurons**, and a neuron is much simpler than its name suggests.

Think of a neuron as a single taster in a tea company. The taster is handed a few measurements about a sample, gives each one a certain amount of importance, adds everything up into one overall impression, and then gives a verdict. The amounts of importance are called **weights**, and they are exactly like the coefficients of the linear regressions you trained in Book 4. The one new ingredient is the verdict step at the end, a small bend that decides how strongly the taster reacts, and Chapter 2 takes all of this apart slowly with real numbers. **A single neuron is nothing more than a weighted sum followed by a small bend, and everything in deep learning is built from that one idea, repeated.**

One taster on their own cannot do very much. The trick is to arrange many of them in **layers**, as Figure 1.2 shows. The first layer of neurons reads the raw inputs, each neuron in the next layer reads every verdict from the layer before it, and the last layer gives the final answer. The layers in the middle are called **hidden layers**, simply because you never see their numbers directly, since they are the network's own private working.

> 🖼️ **Figure 1.2:** A small network for the field. Every line is one weight that training adjusts, and the eight hidden neurons are free to invent whatever in between numbers make the final answer easy.

You can train exactly the network in Figure 1.2 right now, on the raw positions, with no help from you at all. Scikit learn includes a small neural network called `MLPClassifier`, where MLP stands for multilayer perceptron, an old name for this kind of network. You will build the same thing yourself from scratch in later chapters, so for now treat it as a preview.

In [ ]:
from sklearn.neural_network import MLPClassifier

net = MLPClassifier(hidden_layer_sizes=(8,), max_iter=3000,
                    random_state=7)
net.fit(X_train, y_train)
print(round(net.score(X_test, y_test), 2))

The network reached almost the same score as your hand built distance column, but nobody told it that the pattern was a circle. It was given the two raw positions and nothing else, and during training its eight hidden neurons gradually found combinations of those positions that, taken together, carve out the middle of the field. The next cell counts how many numbers the network had to adjust to do it.

In [ ]:
sizes = [w.size for w in net.coefs_] + [b.size for b in net.intercepts_]
print(sum(sizes))

Those 33 numbers are the network's entire knowledge. There are sixteen weights on the lines from the two inputs to the eight hidden neurons, eight weights on the lines from the hidden neurons to the output, and one extra starting number, called a **bias**, for each of the nine neurons that do any work. AlexNet had about 60 million of them, but every one of those millions does the same simple job as one of these 33, which is to say how much one number should count when the next neuron adds things up.

## Why Depth Changes Everything

The network for the field had one hidden layer, and that was enough for a pattern as simple as a circle. The word **deep** in deep learning means that a network has many hidden layers stacked one after another, and the reason this matters so much is that each layer gets to build on the work of the layer before it.

You learned to read in exactly this way. First you learned to recognise the shapes of individual letters, then you learned that certain letters together make a word, and only after that could you follow a sentence and, eventually, the idea behind a whole paragraph. Nobody taught you to understand a paragraph directly from the ink on the page, because that jump is far too large to make in one step. Each stage only had to take a small step beyond the stage before it.

Deep networks that look at photographs turn out to learn in a remarkably similar order. When researchers in 2013 found ways to look inside trained networks like AlexNet, they saw that the first layers had become detectors of simple edges and patches of colour, that the middle layers combined those edges into textures and simple shapes, and that the last layers responded to whole parts of objects, such as eyes, wheels and the faces of dogs. Figure 1.3 shows that climb, and nobody programmed any of it, since every layer arrived at its role purely through training.

> 🖼️ **Figure 1.3:** How a deep network that looks at photographs builds its understanding, one layer at a time. Each step only has to go a little beyond the step below it.

This is the real answer to why AlexNet beat the hand designed rules. The experts had been building the bottom steps of that ladder by hand, writing careful programs to find edges and corners, and then handing those to a classical model. AlexNet built every step itself, including the bottom ones, and because each step was tuned to serve the step above it, the whole ladder fitted together far better than anything a person could design one piece at a time. **Depth lets a network build complicated features out of simpler ones, so it can find patterns in raw data that no person could have described in advance.**

Neural networks were not new in 2012, since the basic ideas had existed for decades. What changed was that three ingredients finally arrived together. The first was enough data, because ImageNet, built by a team led by Fei-Fei Li, offered more than a million labelled photographs at a time when most collections held tens of thousands. The second was fast hardware, because graphics cards built for video games turned out to be extremely good at the kind of arithmetic that networks do. The third was a handful of training tricks, such as a simpler kind of bend inside each neuron and a way of stopping a large network from memorising its training data, and you will meet each of those tricks properly in this book, in Chapters 4 and 15.

It is also worth being honest about what depth does not change. A deep network is still a model trained on examples, it can still overfit, and its test score still means nothing unless it was measured on data the network never saw during training. Everything you learned in Books 4 and 5 about honest validation carries straight across, and this book will lean on it constantly. For a tidy table of house prices, a boosted model from Book 5 is often still the better and cheaper choice, and deep learning earns its place mainly when the data is raw, large and hard to describe with columns.

## What You Will Build in This Book

This book takes you from a single neuron all the way to a complete deep learning project of your own, in six parts. Figure 1.4 shows the route, and each part uses only what the parts before it have taught.

> 🖼️ **Figure 1.4:** The six parts of this book, from a single neuron to a finished project of your own.

The first two parts are built entirely by hand. You will write a neuron in a few lines of numpy, join neurons into a network, measure how wrong it is, and then teach it to improve, using an idea called gradient descent and a method called **backpropagation** that works out how every single weight should change. Doing this by hand is like learning to cook a dish from raw ingredients before you ever buy the ready made sauce. It takes a little longer, but afterwards nothing in the kitchen can confuse you, because you know what every jar on the shelf is really doing.

The third part then hands that same work to **PyTorch**, the framework that much of today's deep learning research is written in. PyTorch will calculate every gradient for you automatically, and Chapter 9 asks you to check that its answers match the ones you worked out by hand, so that you trust the framework because you have tested it, not because somebody told you to. The fourth part teaches the craft of training well, which covers how to choose a learning rate, how to stop a network from memorising its data, and what to do when training simply refuses to work.

The fifth part gives you a first, deliberately short look at the two families of network that power most of modern AI, the convolutional networks that look at images and the sequence models that read text. Each one gets a full book later in the series, Book 7 for images and Book 8 for language, so here the aim is to understand the core idea well enough that those books feel like a natural next step. The last part teaches you to save, load and speed up your networks, and then asks you to carry one complete deep learning project from a raw question all the way to an honest, well explained result.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Shrink the network.** Train the `MLPClassifier` again with `hidden_layer_sizes=(2,)`, so that only two hidden neurons are left. Write down the new test score and the new number of adjustable values, and think about why two neurons might not be enough to carve out a circle.
2. **Make the field harder.** Build the data again with `noise=0.2`, so that the two varieties overlap a little. Compare the logistic regression on the squared distance column with the network on the raw positions, and notice that neither can now be perfect, because some plants really do sit in the wrong place.
3. **Spot raw data around you.** Write down three kinds of data from your own daily life, such as the photos on your phone or the voice notes you send, and for each one try to describe a single useful column you could build by hand. Note how quickly it becomes difficult.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What did AlexNet do differently from the teams that used hand designed rules?**
2. **Why did the logistic regression score only 0.5 on the plants in the field?**
3. **What are the two steps that every neuron performs?**
4. **What does the word "deep" mean in deep learning, and why does it help?**
5. **Name the three ingredients that came together around 2012.**

### Answers

1. It was given only the raw pixels and learned its own features from them, layer by layer, instead of relying on features that experts had designed by hand.
2. Because the two varieties form a circle inside a ring, and no straight line can put a circle on one side and a ring on the other, so the line could do no better than a coin toss.
3. It adds up its inputs, each multiplied by its own weight and with a bias added, and then passes that total through a small bend that decides how strongly it reacts.
4. It means the network has many hidden layers stacked in a row. Each layer builds on the features found by the layer before it, so complicated patterns can be assembled from simple ones.
5. Large labelled datasets such as ImageNet, fast graphics cards for the arithmetic, and a handful of training tricks that made big networks practical to train.

## What Comes Next

You now know why classical machine learning runs out of road on raw data, what a neural network is made of, and why stacking layers lets it find patterns that nobody could describe in advance.

- **Chapter 2, The Artificial Neuron,** opens up a single neuron and builds one in a few lines of numpy, so that you can see every number it produces.
- **Chapter 3, From Neurons to a Network,** joins neurons into layers, runs a whole network forward by hand, and meets the famous puzzle that once convinced researchers that neural networks were a dead end.

Chapter 2 begins in 1958, with a newspaper story claiming that a new kind of learning machine would one day walk, talk and see.